In [1]:
import os
import sys
import csv
import numpy as np
from pathlib import Path
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import (
    EvalCallback, CheckpointCallback, BaseCallback
)
from sumo_rl import SumoEnvironment
from sumo_rl.environment.traffic_signal import TrafficSignal
from sumo_rl.environment.observations import ObservationFunction
from gymnasium import spaces

os.environ["SUMO_HOME"] = "D:\\SUMO"
sys.path.append(os.path.join(os.environ["SUMO_HOME"], "tools"))

# ── Paths ──────────────────────────────────────────────────────────────────────
try:
    base = Path(__file__).resolve().parent.parent
except NameError:
    base = Path.cwd().parent

NET_FILE     = str(base / "nets" / "unrestricted_right_xml_gen" / "unrestricted_right.net.xml")
ROUTE_FILE   = str(base / "nets" / "unrestricted_right_xml_gen" / "unrestricted_right.rou.xml")
OUT_CSV      = str(base / "outputs" / "results" / "PPO" / "sumo_output")
LOG_DIR      = str(base / "outputs" / "logs"    / "PPO")
MODEL_DIR    = str(base / "outputs" / "models"  / "PPO")
CAUSAL_LOG   = str(base / "outputs" / "causal"  / "ppo_causal_log.csv")

for d in [Path(OUT_CSV).parent, LOG_DIR, MODEL_DIR, Path(CAUSAL_LOG).parent]:
    Path(d).mkdir(parents=True, exist_ok=True)

ALGO_NAME       = "PPO"
TOTAL_TIMESTEPS = 50_000
EVAL_FREQ       = 5_000
CHECKPOINT_FREQ = 5_000
N_EVAL_EPISODES = 3
DURATION        = 3600
USE_GUI         = False # Set to False for faster training without rendering


# ── Reward function ────────────────────────────────────────────────────────────
def reward_function(traffic_signal: TrafficSignal) -> float:
    MAX_WAITING   = 300.0
    MAX_QUEUE     = 20.0
    MAX_PRESSURE  = 10.0
    MAX_COLLISION = 1.0

    waiting_time_raw = traffic_signal.get_accumulated_waiting_time_per_lane()
    if isinstance(waiting_time_raw, dict):
        total_waiting = sum(waiting_time_raw.values())
    elif isinstance(waiting_time_raw, (int, float)):
        total_waiting = float(waiting_time_raw)
    else:
        total_waiting = float(sum(waiting_time_raw))

    queue_length    = float(sum(
        traffic_signal.sumo.lane.getLastStepHaltingNumber(lane)
        for lane in traffic_signal.lanes
    ))
    pressure        = float(traffic_signal.get_pressure())
    collision_count = float(len(traffic_signal.sumo.simulation.getCollisions()))

    norm_waiting   = min(total_waiting   / MAX_WAITING,   1.0)
    norm_queue     = min(queue_length    / MAX_QUEUE,     1.0)
    norm_pressure  = min(abs(pressure)   / MAX_PRESSURE,  1.0)
    norm_collision = min(collision_count / MAX_COLLISION, 1.0)

    return -(
        norm_waiting   * 0.3 +
        norm_queue     * 0.3 +
        norm_pressure  * 0.2 +
        norm_collision * 0.2
    )


# ── Observation function ───────────────────────────────────────────────────────
class CustomObservationFunction(ObservationFunction):
    def __init__(self, ts: TrafficSignal):
        super().__init__(ts)

    def __call__(self) -> np.ndarray:
        ts = self.ts
        phase_id = [1 if ts.green_phase == i else 0 for i in range(ts.num_green_phases)]
        density  = [
            ts.sumo.lane.getLastStepVehicleNumber(lane) / (ts.sumo.lane.getLength(lane) / 7.5)
            for lane in ts.lanes
        ]
        queue = [
            ts.sumo.lane.getLastStepHaltingNumber(lane) / (ts.sumo.lane.getLength(lane) / 7.5)
            for lane in ts.lanes
        ]
        speed = [
            ts.sumo.lane.getLastStepMeanSpeed(lane) / max(ts.sumo.lane.getMaxSpeed(lane), 1e-6)
            for lane in ts.lanes
        ]
        return np.clip(np.array(phase_id + density + queue + speed, dtype=np.float32), 0.0, 1.0)

    def observation_space(self) -> spaces.Box:
        size = self.ts.num_green_phases + 3 * len(self.ts.lanes)
        return spaces.Box(low=0.0, high=1.0, shape=(size,), dtype=np.float32)


# ── Causal variable logger ─────────────────────────────────────────────────────
class CausalLogger:
    """
    Collects per-step causal variables directly from the SUMO environment.
    Writes to CSV after every episode for use in SCM fitting.

    Variables logged and their causal role:
        Interventions : phase_action
        Mediators     : mean_speed, queue_length, pressure
        Outcomes      : waiting_time, collision_count
        Context       : step, episode, reward
    """

    FIELDS = [
        "episode", "step", "reward",
        # Intervention
        "phase_action",
        # Mediators
        "mean_speed", "queue_length", "pressure",
        # Outcomes
        "waiting_time", "collision_count",
    ]

    def __init__(self, csv_path: str):
        self.csv_path = csv_path
        self.buffer   = []
        self.episode  = 0
        self._init_csv()

    def _init_csv(self):
        with open(self.csv_path, "w", newline="") as f:
            csv.DictWriter(f, fieldnames=self.FIELDS).writeheader()

    def log(self, env: SumoEnvironment, action: int, reward: float, step: int):
        ts_id = env.ts_ids[0]
        ts    = env.traffic_signals[ts_id]

        # ── Intervention ──────────────────────────────────────────────────────
        phase_action = int(action)

        # ── Mediators ─────────────────────────────────────────────────────────
        mean_speed = float(np.mean([
            ts.sumo.lane.getLastStepMeanSpeed(lane)
            for lane in ts.lanes
        ]))

        queue_length = float(sum(
            ts.sumo.lane.getLastStepHaltingNumber(lane)
            for lane in ts.lanes
        ))

        # ── Pressure — handle all return types ────────────────────────────────
        pressure_raw = ts.get_pressure()
        if isinstance(pressure_raw, list):
            pressure = float(sum(pressure_raw))
        elif isinstance(pressure_raw, dict):
            pressure = float(sum(pressure_raw.values()))
        else:
            pressure = float(pressure_raw)

        # ── Outcomes ──────────────────────────────────────────────────────────
        waiting_time_raw = ts.get_accumulated_waiting_time_per_lane()
        if isinstance(waiting_time_raw, dict):
            waiting_time = float(sum(waiting_time_raw.values()))
        elif isinstance(waiting_time_raw, (int, float)):
            waiting_time = float(waiting_time_raw)
        else:
            waiting_time = float(sum(waiting_time_raw))

        collision_count = int(len(ts.sumo.simulation.getCollisions()))

        self.buffer.append({
            "episode"        : self.episode,
            "step"           : step,
            "reward"         : round(reward,        6),
            "phase_action"   : phase_action,
            "mean_speed"     : round(mean_speed,    4),
            "queue_length"   : round(queue_length,  2),
            "pressure"       : round(pressure,      4),
            "waiting_time"   : round(waiting_time,  2),
            "collision_count": collision_count,
        })

    def flush(self):
        """Write buffer to CSV and clear it. Call at episode end."""
        if not self.buffer:
            return
        with open(self.csv_path, "a", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=self.FIELDS)
            writer.writerows(self.buffer)
        self.buffer = []
        self.episode += 1


# ── Causal callback ────────────────────────────────────────────────────────────
class CausalLoggingCallback(BaseCallback):
    """
    Hooks into SB3 training loop to extract causal variables each step.
    Accesses the raw SumoEnvironment inside DummyVecEnv.
    """

    def __init__(self, causal_logger: CausalLogger, verbose=0):
        super().__init__(verbose)
        self.causal_logger = causal_logger
        self.step_count    = 0

    def _get_raw_env(self) -> SumoEnvironment:
        """Unwrap DummyVecEnv → Monitor → SumoEnvironment."""
        return self.training_env.envs[0].env

    def _on_step(self) -> bool:
        raw_env = self._get_raw_env()
        action  = self.locals["actions"][0]
        reward  = self.locals["rewards"][0]

        self.causal_logger.log(raw_env, action, reward, self.step_count)
        self.step_count += 1

        # Flush to CSV at episode end
        if self.locals["dones"][0]:
            self.causal_logger.flush()

        return True


# ── Environment factory ────────────────────────────────────────────────────────
def make_env(out_csv_name: str):
    def _init():
        env = SumoEnvironment(
            net_file=NET_FILE,
            route_file=ROUTE_FILE,
            out_csv_name=out_csv_name,
            single_agent=True,
            use_gui=USE_GUI,
            num_seconds=DURATION,
            delta_time=5,
            yellow_time=2,
            min_green=5,
            reward_fn=reward_function,
            observation_class=CustomObservationFunction,
            sumo_warnings=False,
        )
        return Monitor(env, LOG_DIR)
    return _init


# ── Main ───────────────────────────────────────────────────────────────────────
if __name__ == "__main__":

    assert Path(NET_FILE).exists(),   f"net_file not found:   {NET_FILE}"
    assert Path(ROUTE_FILE).exists(), f"route_file not found: {ROUTE_FILE}"

    print(f"\n{'='*55}")
    print(f"  Algorithm  : {ALGO_NAME}")
    print(f"  Timesteps  : {TOTAL_TIMESTEPS:,}")
    print(f"  Causal log : {CAUSAL_LOG}")
    print(f"{'='*55}\n")

    causal_logger = CausalLogger(CAUSAL_LOG)

    train_env = DummyVecEnv([make_env(OUT_CSV + "_train")])
    eval_env  = DummyVecEnv([make_env(OUT_CSV + "_eval")])

    callbacks = [
        CausalLoggingCallback(causal_logger),        # ← logs causal vars
        CheckpointCallback(
            save_freq=CHECKPOINT_FREQ,
            save_path=MODEL_DIR,
            name_prefix=ALGO_NAME,
            verbose=1,
        ),
        EvalCallback(
            eval_env,
            best_model_save_path=MODEL_DIR,
            log_path=LOG_DIR,
            eval_freq=EVAL_FREQ,
            n_eval_episodes=N_EVAL_EPISODES,
            deterministic=True,
            verbose=1,
        ),
    ]

    model = PPO(
        policy="MlpPolicy",
        env=train_env,
        tensorboard_log=LOG_DIR,
        verbose=1,
        learning_rate=3e-4,
        n_steps=512,
        batch_size=64,
        n_epochs=10,
        gamma=0.99,
        gae_lambda=0.95,
        ent_coef=0.01,
    )

    model.learn(
        total_timesteps=TOTAL_TIMESTEPS,
        callback=callbacks,
        tb_log_name=ALGO_NAME
    )

    final_path = str(Path(MODEL_DIR) / f"{ALGO_NAME}_final")
    model.save(final_path)
    print(f"\n✓ Training complete → {final_path}")
    print(f"✓ Causal log saved  → {CAUSAL_LOG}")

    train_env.close()
    eval_env.close()


  Algorithm  : PPO
  Timesteps  : 50,000
  Causal log : d:\Robotic_Sensing_and Transfer_Learning\Simulation\outputs\causal\ppo_causal_log.csv

Using cpu device
Logging to d:\Robotic_Sensing_and Transfer_Learning\Simulation\outputs\logs\PPO\PPO_7
----------------------------
| time/              |     |
|    fps             | 55  |
|    iterations      | 1   |
|    time_elapsed    | 9   |
|    total_timesteps | 512 |
----------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 720          |
|    ep_rew_mean          | -76.8        |
| time/                   |              |
|    fps                  | 52           |
|    iterations           | 2            |
|    time_elapsed         | 19           |
|    total_timesteps      | 1024         |
| train/                  |              |
|    approx_kl            | 0.0076828175 |
|    clip_fraction        | 0.0334       |
|    clip_range           | 0.2   

KeyboardInterrupt: 

In [4]:
train_env.close()
eval_env.close()